##Step 1: Load and chunk pdf

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(Path.cwd() / ".env")  # pulls ANTHROPIC_API_KEY and VOYAGE_API_KEY from your .env file

True

##Step 2:Load PDFs

In [5]:
data_dir = Path("data")
pdf_paths = sorted(data_dir.glob("*.pdf"))

print(f"Found {len(pdf_paths)} PDF(s):")
for p in pdf_paths:
    print(" -", p.name)

documents = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

print(f"\nLoaded {len(documents)} pages total across all PDFs")

Found 3 PDF(s):
 - NIST.AI.100-1.pdf
 - owasp-top-10.pdf
 - RP-008344-DS-5-raspberry-pi-4-product-brief.pdf

Loaded 96 pages total across all PDFs


## step 3: Chunk the PDF's

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # max characters per chunk
    chunk_overlap=150,   # overlap so a sentence split across a chunk boundary isn't lost
    separators=["\n\n", "\n", ". ", " ", ""]  # prefers paragraph breaks, falls back to smaller splits
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 205


##Step 4:Perform chunk analysis and provide chunk metrics

In [9]:
chunk_lengths = [len(c.page_content) for c in chunks]
avg_chunk_size = sum(chunk_lengths) / len(chunk_lengths)

print(f"Total number of chunks: {len(chunks)}")
print(f"Average chunk size: {avg_chunk_size:.1f} characters")
print(f"Smallest chunk: {min(chunk_lengths)} characters")
print(f"Largest chunk: {max(chunk_lengths)} characters")

print("\n--- Sample chunk ---")
print(chunks[0].page_content[:300])
print("\nMetadata:", chunks[0].metadata)

Total number of chunks: 205
Average chunk size: 746.3 characters
Smallest chunk: 14 characters
Largest chunk: 1000 characters

--- Sample chunk ---
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)

Metadata: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-24T14:45:46-05:00', 'author': 'National Institute of Standards and Technology', 'keywords': 'Artificial Intelligence (AI); AI; AI RMF; AI RMF 1.0; AI systems; trustworthy and responsible AI.', 'moddate': '2025-06-04T13:01:45-04:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': 'As directed by the National Artificial Intelligence Initiative Act of 2020 (P.L. 116-283), the goal of the AI RMF is to offer a resource to the organizations designing, developing, deploying, or using AI systems to help manage the many risks of AI and promote trustworthy and responsible development and use of AI system

##Step 5: Vector embeddings and FAISS index built

In [10]:
from langchain_voyageai import VoyageAIEmbeddings
from langchain_community.vectorstores.faiss import FAISS

embeddings = VoyageAIEmbeddings(model="voyage-4")  # reads VOYAGE_API_KEY automatically

vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS vectorstore built with {vectorstore.index.ntotal} vectors")

FAISS vectorstore built with 205 vectors


##Step6: Save FAISS index to disk

In [11]:
vectorstore.save_local("faiss_index")
print("Vectorstore saved to ./faiss_index")

Vectorstore saved to ./faiss_index


##Step7:Retrieval Query

In [12]:
query = "What are the key risks associated with deploying AI systems?"

results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} (source: {doc.metadata.get('source')}, page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:400])
    print()

Query: What are the key risks associated with deploying AI systems?

--- Result 1 (source: data\NIST.AI.100-1.pdf, page 10) ---
All involved AI actors share responsibilities for designing, developing, and deploying a
trustworthy AI system that is fit for purpose.
Risk in real-world settings: While measuring AI risks in a laboratory or a controlled
environment may yield important insights pre-deployment, these measurements may differ
from risks that emerge in operational, real-world settings.
Inscrutability: Inscrutable AI 

--- Result 2 (source: data\NIST.AI.100-1.pdf, page 12) ---
across other types of software development and deployment. Examples of overlapping risks
include: privacy concerns related to the use of underlying data to train AI systems; the en-
ergy and environmental implications associated with resource-heavy computing demands;
security concerns related to the confidentiality, integrity, and availability of the system and
its training and output data; and ge

--- Resul

##Step 8:Answer generation with Claude

In [13]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model_name="claude-sonnet-5")  # reads ANTHROPIC_API_KEY automatically

context = "\n\n".join(doc.page_content for doc in results)

prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {query}

Answer:"""

response = llm.invoke(prompt)
print(response.content)

Based on the provided context, the key risks associated with deploying AI systems include:

1. **Risk measurement discrepancies in real-world settings**: Measurements of AI risks taken in laboratory or controlled environments may differ from risks that emerge in operational, real-world settings.

2. **Inscrutability**: This can complicate risk measurement and stems from:
   - The opaque nature of AI systems (limited explainability or interpretability)
   - Lack of transparency or documentation in AI system development or deployment
   - Inherent uncertainties in AI systems

3. **Human baseline challenges**: For AI systems intended to augment or replace human activity (e.g., decision making), establishing baseline metrics for comparison is difficult since AI systems carry out different tasks—and perform tasks differently—than humans.

4. **Overlapping risks with other software development/deployment**, including:
   - Privacy concerns related to the use of underlying data to train AI sy